In [ ]:
"""
================================================================================
CRÉDITO, POLÍTICA MONETÁRIA E INADIMPLÊNCIA DAS MPE NO BRASIL (2012–2026)
Pipeline Preditivo e Estocástico com Validação Out-of-Time (OOT)
Autor: Itaiguara de Oliveira Bezerra
================================================================================
Metodologia:
  1. Extração e agregação setorial/regional dos microdados SCR/BCB.
  2. Coleta das séries macrofinanceiras via API SGS/BCB.
  3. Calibração endógena do limiar de estresse via Persistência de Markov (P11 >= 0.70).
  4. Treinamento supervisionado (2013-2021) e teste Out-of-Time (2022-2026).
  5. Avaliação de métricas probabilísticas e discriminatórias (ROC-AUC, PR-AUC, Brier).
  6. Interpretabilidade econômica e direcional via Tree SHAP.
================================================================================
"""

from typing import Dict, List, Tuple
import warnings
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import shap

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False

warnings.filterwarnings('ignore')

# Configuração estética global dos gráficos
plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)

CORES_SETORES = {
    'Comercio': '#e377c2',
    'Construcao': '#2ca02c',
    'Ind_Transformacao': '#1f77b4',
    'Servicos': '#ff7f0e',
}

MAPA_REGIOES = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste',
    'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}


def remover_molduras(ax: plt.Axes = None) -> None:
    """Remove as bordas externas do gráfico para layout limpo."""
    if ax is None:
        ax = plt.gca()
    for spine in ['top', 'right', 'left', 'bottom']:
        ax.spines[spine].set_visible(False)


def carregar_dados_scr(caminho_base: str = 'painel_scr_mpe_2012_2026') -> pd.DataFrame:
    """Carrega a base bruta do SCR em formato Parquet ou CSV."""
    try:
        df = pd.read_parquet(f'{caminho_base}.parquet')
    except Exception:
        df = pd.read_csv(f'{caminho_base}.csv', sep=';', decimal=',')
    
    col_data = 'data' if 'data' in df.columns else 'data_base'
    df['data'] = pd.to_datetime(df[col_data])
    return df


def consultar_sgs(codigo_serie: int, nome_coluna: str) -> pd.DataFrame:
    """Consulta a API de dados abertos do SGS/BCB."""
    url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo_serie}/dados?formato=json'
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            df = pd.DataFrame(r.json())
            df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
            df[nome_coluna] = df['valor'].astype(float)
            df['ano_mes'] = df['data'].dt.to_period('M')
            return df[['ano_mes', nome_coluna]].drop_duplicates(subset=['ano_mes'])
    except Exception as e:
        print(f'[AVISO] Falha ao consultar série SGS {codigo_serie}: {e}')
    
    raise ConnectionError(f'Não foi possível extrair a série SGS {codigo_serie}.')


def plotar_perfil_descritivo(df_painel: pd.DataFrame, caminho_saida: str = 'perfil_descritivo_base_mpe.png') -> None:
    """Gera o painel com o histórico agregado, trajetória setorial e boxplot."""
    fig = plt.figure(figsize=(15, 10))
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.28, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0:2])
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax3 = fig.add_subplot(gs[1, 1:3])

    # 1. Carteira Total
    df_tempo = df_painel.groupby('data')['carteira_ativa'].sum() / 1e9
    ax1.plot(df_tempo.index, df_tempo.values, color='#1f77b4', lw=2.5)
    ax1.set_ylabel('R$ Bilhões', fontweight='bold')
    ax1.set_xlabel('Ano', fontweight='bold')
    remover_molduras(ax1)

    # 2. Inadimplência Setorial
    for setor in sorted(df_painel['setor_macro'].unique()):
        subset = df_painel[df_painel['setor_macro'] == setor]
        ax2.plot(
            subset['data'],
            subset['taxa_inadimplencia_pct'],
            lw=2.2,
            label=setor,
            color=CORES_SETORES.get(setor, '#333333'),
        )
    ax2.set_ylabel('Inadimplência (%)', fontweight='bold')
    ax2.set_xlabel('Ano', fontweight='bold')
    ax2.legend(loc='best', frameon=False, fontsize=9.5)
    remover_molduras(ax2)

    # 3. Boxplot Setorial
    sns.boxplot(
        data=df_painel,
        x='setor_macro',
        y='taxa_inadimplencia_pct',
        hue='setor_macro',
        palette='Blues',
        ax=ax3,
        legend=False,
        width=0.55,
    )
    ax3.set_xlabel('Setor Macro', fontweight='bold')
    ax3.set_ylabel('Inadimplência (%)', fontweight='bold')
    remover_molduras(ax3)

    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'[OK] Gráfico exportado: {caminho_saida}')


def plotar_painel_regional(df_raw: pd.DataFrame, caminho_saida: str = 'mapa_regional_setorial_mpe.png') -> None:
    """Gera o mapa cloroplético regional com os rankings setoriais por macrorregião."""
    df_geo = df_raw.copy()
    if 'regiao' not in df_geo.columns and 'uf' in df_geo.columns:
        df_geo['regiao'] = df_geo['uf'].map(MAPA_REGIOES)

    resumo_reg_total = (
        df_geo.groupby('regiao')
        .agg(carteira_total=('carteira_ativa', 'sum'), vencido_total=('vencido_acima_90', 'sum'))
        .reset_index()
    )
    resumo_reg_total['inadimplencia_total_pct'] = (
        resumo_reg_total['vencido_total'] / resumo_reg_total['carteira_total']
    ) * 100

    resumo_reg_setor = (
        df_geo.groupby(['regiao', 'setor_macro'])
        .agg(carteira_setor=('carteira_ativa', 'sum'), vencido_setor=('vencido_acima_90', 'sum'))
        .reset_index()
    )
    resumo_reg_setor['inadimplencia_setor_pct'] = (
        resumo_reg_setor['vencido_setor'] / resumo_reg_setor['carteira_setor']
    ) * 100

    fig_mapa = plt.figure(figsize=(19, 11), dpi=300)
    gs_geo = gridspec.GridSpec(5, 2, figure=fig_mapa, width_ratios=[1.15, 0.85], wspace=0.25, hspace=0.40)
    ax_mapa = fig_mapa.add_subplot(gs_geo[:, 0])

    mapa_plotado = False
    if HAS_GEOPANDAS:
        try:
            url_geojson = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'
            resp_geo = requests.get(url_geojson, headers={'User-Agent': 'Mozilla/5.0'}, timeout=25)
            if resp_geo.status_code == 200:
                gdf_estados = gpd.read_file(resp_geo.text)
                gdf_estados['regiao'] = gdf_estados['sigla'].map(MAPA_REGIOES)
                gdf_regioes = gdf_estados.dissolve(by='regiao', as_index=False)
                gdf_final = gdf_regioes.merge(resumo_reg_total, on='regiao', how='left')

                gdf_final.plot(
                    column='inadimplencia_total_pct',
                    cmap='YlOrRd',
                    linewidth=1.2,
                    edgecolor='#4a4a4a',
                    legend=True,
                    legend_kwds={
                        'label': 'Taxa Agregada de Inadimplência MPE (> 90 dias) %',
                        'orientation': 'horizontal',
                        'shrink': 0.65,
                        'pad': 0.02,
                    },
                    ax=ax_mapa,
                )

                offset_coords = {
                    'Norte': (-56.0, -3.5),
                    'Nordeste': (-40.5, -8.5),
                    'Centro-Oeste': (-54.0, -15.5),
                    'Sudeste': (-43.5, -20.5),
                    'Sul': (-52.0, -28.0),
                }

                for _, r_row in gdf_final.iterrows():
                    reg_nome = r_row['regiao']
                    taxa_val = r_row['inadimplencia_total_pct']
                    x_c, y_c = offset_coords.get(reg_nome, (r_row.geometry.centroid.x, r_row.geometry.centroid.y))
                    ax_mapa.annotate(
                        text=f'{reg_nome}\n{taxa_val:.2f}%',
                        xy=(x_c, y_c),
                        ha='center',
                        va='center',
                        fontsize=10.5,
                        fontweight='bold',
                        color='#111111',
                        bbox=dict(boxstyle='round,pad=0.35', fc='#ffffff', ec='#777777', lw=0.8, alpha=0.9),
                    )
                ax_mapa.set_axis_off()
                mapa_plotado = True
        except Exception as e_mapa:
            print(f'[AVISO] Renderização GeoPandas: {e_mapa}')

    if not mapa_plotado:
        df_alt = resumo_reg_total.sort_values('inadimplencia_total_pct', ascending=True)
        bars_alt = ax_mapa.barh(df_alt['regiao'], df_alt['inadimplencia_total_pct'], color='#d62728', height=0.55)
        for b in bars_alt:
            w = b.get_width()
            ax_mapa.text(w + 0.05, b.get_y() + b.get_height() / 2, f'{w:.2f}%', va='center', ha='left', fontsize=10, fontweight='bold')
        ax_mapa.set_xlabel('Inadimplência (> 90 dias) %', fontweight='bold')
        remover_molduras(ax_mapa)

    ordem_regioes = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
    for i, reg_nome in enumerate(ordem_regioes):
        ax_sub = fig_mapa.add_subplot(gs_geo[i, 1])
        sub_setor = resumo_reg_setor[resumo_reg_setor['regiao'] == reg_nome].sort_values('inadimplencia_setor_pct', ascending=True)
        
        if len(sub_setor) > 0:
            cores_barras = [CORES_SETORES.get(s, '#1f77b4') for s in sub_setor['setor_macro']]
            bars_reg = ax_sub.barh(sub_setor['setor_macro'], sub_setor['inadimplencia_setor_pct'], color=cores_barras, height=0.6)
            max_v = sub_setor['inadimplencia_setor_pct'].max()
            for b_reg in bars_reg:
                w_reg = b_reg.get_width()
                ax_sub.text(w_reg + (max_v * 0.02 if max_v > 0 else 0.05), b_reg.get_y() + b_reg.get_height() / 2, f'{w_reg:.2f}%', va='center', ha='left', fontsize=8.5, fontweight='bold', color='#222222')
            ax_sub.set_xlim(0, max_v * 1.25 if max_v > 0 else 1.0)
        
        ax_sub.set_title(f'Setores em {reg_nome}', fontsize=9.5, fontweight='bold', pad=4)
        ax_sub.tick_params(axis='y', labelsize=8.5)
        ax_sub.tick_params(axis='x', labelsize=8)
        remover_molduras(ax_sub)

    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'[OK] Gráfico exportado: {caminho_saida}')


def otimizar_limiar_markov(df_setor_tr: pd.DataFrame, min_p11: float = 0.70) -> Tuple[float, float, float]:
    """Calcula o corte ótimo tau_i onde a cadeia de Markov satisfaz P11 >= min_p11."""
    serie = df_setor_tr['taxa_inadimplencia_pct'].values
    percentis = np.linspace(50, 85, 36)
    candidatos_tau = np.percentile(serie, percentis)
    
    melhor_tau = None
    melhor_p11 = 0.0
    melhor_p00 = 0.0
    
    for tau in candidatos_tau:
        estados = (serie >= tau).astype(int)
        counts = np.zeros((2, 2))
        for t in range(len(estados) - 1):
            counts[estados[t], estados[t+1]] += 1
        
        row_sums = counts.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        P = counts / row_sums
        
        p00, p11 = P[0, 0], P[1, 1]
        
        if p11 >= min_p11 and p00 >= 0.70:
            if p11 > melhor_p11:
                melhor_p11 = p11
                melhor_p00 = p00
                melhor_tau = tau
                
    if melhor_tau is None:
        melhor_tau = float(np.median(serie))
        melhor_p11 = 0.50
        melhor_p00 = 0.50
        
    return float(melhor_tau), float(melhor_p00), float(melhor_p11)


def treinar_e_avaliar_modelos(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    prevalencia_oot: float
) -> Tuple[Dict, pd.DataFrame, GradientBoostingClassifier]:
    """Executa o treinamento dos quatro classificadores e gera os gráficos de desempenho."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 1. Regressão Logística
    log_reg = LogisticRegression(random_state=42, max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)
    y_pred_lr, y_prob_lr = log_reg.predict(X_test_scaled), log_reg.predict_proba(X_test_scaled)[:, 1]

    # 2. Árvore de Decisão
    dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
    dt.fit(X_train, y_train)
    y_pred_dt, y_prob_dt = dt.predict(X_test), dt.predict_proba(X_test)[:, 1]

    # 3. Random Forest
    rf = RandomForestClassifier(n_estimators=150, max_depth=4, min_samples_leaf=5, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf, y_prob_rf = rf.predict(X_test), rf.predict_proba(X_test)[:, 1]

    # 4. Gradient Boosting
    gbm = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
    gbm.fit(X_train, y_train)
    y_pred_gbm, y_prob_gbm = gbm.predict(X_test), gbm.predict_proba(X_test)[:, 1]

    modelos_dict = {
        '1. Regressão Logística (Baseline)': (y_pred_lr, y_prob_lr),
        '2. Árvore de Decisão': (y_pred_dt, y_prob_dt),
        '3. Random Forest': (y_pred_rf, y_prob_rf),
        '4. Gradient Boosting': (y_pred_gbm, y_prob_gbm),
    }

    # Gráfico: Matrizes de Confusão
    fig, axes = plt.subplots(2, 2, figsize=(10, 8.5))
    plots_config = [
        ('1. Regressão Logística (Baseline)', y_pred_lr, 'Purples', axes[0, 0]),
        ('2. Árvore de Decisão', y_pred_dt, 'Blues', axes[0, 1]),
        ('3. Random Forest', y_pred_rf, 'Oranges', axes[1, 0]),
        ('4. Gradient Boosting', y_pred_gbm, 'Greens', axes[1, 1]),
    ]

    for subtitulo, pred, cmap, ax in plots_config:
        cm = confusion_matrix(y_test, pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax, cbar=False, annot_kws={'size': 13, 'weight': 'bold'})
        ax.set_title(subtitulo, fontsize=10.5, fontweight='bold', pad=8)
        ax.set_xlabel('Previsto', fontweight='bold', fontsize=9.5)
        ax.set_ylabel('Real', fontweight='bold', fontsize=9.5)
        ax.set_xticklabels(['Baixo (0)', 'Alto (1)'])
        ax.set_yticklabels(['Baixo (0)', 'Alto (1)'])

    fig.tight_layout()
    plt.savefig('lab4_matrizes_confusao_oot.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Gráfico: Curva ROC
    plt.figure(figsize=(8.5, 6))
    cores_modelos = ['#9467bd', '#1f77b4', '#ff7f0e', '#2ca02c']
    for (nome, (_, prob)), cor in zip(modelos_dict.items(), cores_modelos):
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc_val = roc_auc_score(y_test, prob)
        plt.plot(fpr, tpr, label=f'{nome} (AUC = {auc_val:.4f})', color=cor, lw=2.2)

    plt.plot([0, 1], [0, 1], color='gray', linestyle=':', lw=1.5, label='Classificador Aleatório (AUC = 0.5000)')
    plt.xlabel('Taxa de Falsos Positivos (1 - Especificidade)', fontsize=10, fontweight='bold')
    plt.ylabel('Taxa de Verdadeiros Positivos (Sensibilidade / Recall)', fontsize=10, fontweight='bold')
    plt.legend(loc='lower right', frameon=False, fontsize=9.5)
    remover_molduras()
    plt.tight_layout()
    plt.savefig('lab4_curva_roc_oot.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Gráfico: Curva PR
    plt.figure(figsize=(8.5, 6))
    for (nome, (_, prob)), cor in zip(modelos_dict.items(), cores_modelos):
        precision, recall, _ = precision_recall_curve(y_test, prob)
        pr_auc_val = average_precision_score(y_test, prob)
        plt.plot(recall, precision, label=f'{nome} (PR-AUC = {pr_auc_val:.4f})', color=cor, lw=2.2)

    plt.axhline(y=prevalencia_oot, color='gray', linestyle=':', lw=1.5, label=f'Baseline Sem Habilidade (Prevalência = {prevalencia_oot:.2%})')
    plt.xlabel('Recall (Sensibilidade)', fontsize=10, fontweight='bold')
    plt.ylabel('Precisão (Valor Preditivo Positivo)', fontsize=10, fontweight='bold')
    plt.ylim([0.0, 1.05])
    plt.xlim([0.0, 1.0])
    plt.legend(loc='upper right', frameon=False, fontsize=9.5)
    remover_molduras()
    plt.tight_layout()
    plt.savefig('lab4_curva_pr_oot.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Tabela consolidada de métricas
    tabela_metricas = []
    for nome, (pred, prob) in modelos_dict.items():
        acc = accuracy_score(y_test, pred)
        roc = roc_auc_score(y_test, prob)
        pr = average_precision_score(y_test, prob)
        brier = brier_score_loss(y_test, prob)
        tabela_metricas.append({
            'Modelo': nome,
            'Acurácia': f'{acc:.2%}',
            'ROC-AUC': f'{roc:.4f}',
            'PR-AUC': f'{pr:.4f}',
            'Brier Score': f'{brier:.4f}'
        })

    df_metricas = pd.DataFrame(tabela_metricas)
    df_metricas.to_excel('tabela_modelos_oot_mpe.xlsx', index=False)
    return modelos_dict, df_metricas, gbm


def calcular_e_plotar_shap(gbm_model: GradientBoostingClassifier, X_test: pd.DataFrame) -> None:
    """Calcula os valores SHAP direcionais agregados e setoriais."""
    explainer = shap.TreeExplainer(gbm_model)
    shap_values = explainer.shap_values(X_test)
    feature_names = X_test.columns.tolist()

    correlacoes_geral = []
    for i, col in enumerate(feature_names):
        val_feat = X_test[col].values
        val_shap = shap_values[:, i]
        if np.std(val_feat) > 1e-6 and np.std(val_shap) > 1e-6:
            corr = np.corrcoef(val_feat, val_shap)[0, 1]
        else:
            corr = 0.0
        correlacoes_geral.append(corr)

    mean_abs_shap_geral = np.abs(shap_values).mean(axis=0)
    cores_geral = ['#d62728' if c >= 0 else '#1f77b4' for c in correlacoes_geral]

    df_shap_geral = pd.DataFrame({
        'feature': feature_names,
        'importance': mean_abs_shap_geral,
        'cor': cores_geral,
    }).sort_values('importance', ascending=True)

    patch_risco = mpatches.Patch(color='#d62728', label='Impacto Negativo (-): Associação Positiva com Risco')
    patch_protetivo = mpatches.Patch(color='#1f77b4', label='Impacto Positivo (+): Associação Negativa com Risco')

    # Gráfico SHAP Geral
    plt.figure(figsize=(11, 6), dpi=300)
    bars_geral = plt.barh(
        df_shap_geral['feature'],
        df_shap_geral['importance'],
        color=df_shap_geral['cor'],
        edgecolor='black',
        alpha=0.85,
        height=0.70,
    )
    max_val_geral = df_shap_geral['importance'].max()
    for bar in bars_geral:
        w = bar.get_width()
        plt.text(w + (max_val_geral * 0.015), bar.get_y() + bar.get_height() / 2, f'{w:.2f}', va='center', ha='left', fontsize=9.5, fontweight='bold', color='#333333')

    plt.legend(handles=[patch_risco, patch_protetivo], loc='lower right', frameon=True, facecolor='white', edgecolor='none', fontsize=9.0)
    plt.xlim(0, max_val_geral * 1.18)
    plt.xlabel('Impacto Médio Absoluto (|SHAP|)', fontsize=10, fontweight='bold')
    remover_molduras()
    plt.tight_layout()
    plt.savefig('lab4_shap_bar_geral_oot_ajustado.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Gráfico SHAP Setorial (2x2)
    idx_comercio = (X_test['setor_macro_Comercio'] == 1).values if 'setor_macro_Comercio' in X_test else np.zeros(len(X_test), dtype=bool)
    idx_construcao = (X_test['setor_macro_Construcao'] == 1).values if 'setor_macro_Construcao' in X_test else np.zeros(len(X_test), dtype=bool)
    idx_industria = (X_test['setor_macro_Ind_Transformacao'] == 1).values if 'setor_macro_Ind_Transformacao' in X_test else np.zeros(len(X_test), dtype=bool)
    idx_servicos = (X_test['setor_macro_Servicos'] == 1).values if 'setor_macro_Servicos' in X_test else np.zeros(len(X_test), dtype=bool)

    features_continuas = [f for f in feature_names if not f.startswith('setor_macro_')]
    indices_features = [feature_names.index(f) for f in features_continuas]

    fig_shap, axes_shap = plt.subplots(2, 2, figsize=(17, 11), dpi=300)
    axes_shap = axes_shap.flatten()

    setores_shap = [
        (axes_shap[0], 'SHAP: Comércio', idx_comercio),
        (axes_shap[1], 'SHAP: Construção Civil', idx_construcao),
        (axes_shap[2], 'SHAP: Indústria de Transformação', idx_industria),
        (axes_shap[3], 'SHAP: Serviços', idx_servicos),
    ]

    for ax, subtitulo, mask_setor in setores_shap:
        if mask_setor.sum() > 0:
            shap_subset = shap_values[mask_setor, :]
            mean_shap_setor = np.abs(shap_subset[:, indices_features]).mean(axis=0)

            cores_setor = []
            for f_idx, col in zip(indices_features, features_continuas):
                val_f = X_test.loc[mask_setor, col].values
                val_s = shap_subset[:, f_idx]
                corr = np.corrcoef(val_f, val_s)[0, 1] if (np.std(val_f) > 1e-6 and np.std(val_s) > 1e-6) else 0.0
                cores_setor.append('#d62728' if corr >= 0 else '#1f77b4')
        else:
            mean_shap_setor = np.zeros(len(features_continuas))
            cores_setor = ['#1f77b4'] * len(features_continuas)

        df_setor_shap = pd.DataFrame({
            'feature': features_continuas,
            'importance': mean_shap_setor,
            'cor': cores_setor,
        }).sort_values('importance', ascending=True)

        bars = ax.barh(df_setor_shap['feature'], df_setor_shap['importance'], color=df_setor_shap['cor'], edgecolor='black', alpha=0.85, height=0.68)
        max_val = df_setor_shap['importance'].max() if df_setor_shap['importance'].max() > 0 else 1.0
        for bar in bars:
            w = bar.get_width()
            ax.text(w + (max_val * 0.025), bar.get_y() + bar.get_height() / 2, f'{w:.1f}', va='center', ha='left', fontsize=9.0, fontweight='bold', color='#333333')

        ax.set_xlim(0, max_val * 1.35)
        ax.set_title(subtitulo, fontsize=10.5, fontweight='bold', pad=8)
        ax.set_xlabel('Impacto Médio (|SHAP|)', fontsize=9.0, fontweight='bold')
        ax.legend(handles=[patch_risco, patch_protetivo], loc='lower right', frameon=True, facecolor='white', edgecolor='none', fontsize=8.0)
        remover_molduras(ax)

    plt.tight_layout()
    plt.savefig('lab4_shap_por_setor_oot_ajustado.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('[OK] Gráficos SHAP exportados com sucesso.')


def main() -> None:
    print('=' * 80)
    print('PIPELINE ECONOMÉTRICO E DE MACHINE LEARNING: INADIMPLÊNCIA MPE (2012–2026)')
    print('=' * 80)

    # 1. Carregamento dos dados
    print('\n[1/5] Carregando microdados do SCR e consolidando setores...')
    df_raw = carregar_dados_scr()
    df_painel = (
        df_raw.groupby(['data', 'setor_macro'])
        .agg(
            carteira_ativa=('carteira_ativa', 'sum'),
            vencido_acima_90=('vencido_acima_90', 'sum'),
            numero_de_operacoes=('numero_de_operacoes', 'sum'),
        )
        .reset_index()
    )
    df_painel['taxa_inadimplencia_pct'] = np.where(
        df_painel['carteira_ativa'] > 0,
        (df_painel['vencido_acima_90'] / df_painel['carteira_ativa']) * 100,
        0.0,
    )
    df_painel['ano_mes'] = df_painel['data'].dt.to_period('M')

    # 2. Diagnóstico descritivo e geográfico
    print('\n[2/5] Gerando gráficos de perfil histórico e distribuição regional...')
    plotar_perfil_descritivo(df_painel)
    plotar_painel_regional(df_raw)

    # 3. Covariáveis e séries macrofinanceiras
    print('\n[3/5] Estruturando defasagens temporais e consultando SGS/BCB...')
    df_painel = df_painel.sort_values(['setor_macro', 'data']).reset_index(drop=True)
    df_painel['ticket_medio_mil'] = (df_painel['carteira_ativa'] / df_painel['numero_de_operacoes']) / 1000.0
    df_painel['crescimento_carteira_interanual'] = df_painel.groupby('setor_macro')['carteira_ativa'].pct_change(12) * 100
    df_painel['crescimento_operacoes_interanual'] = df_painel.groupby('setor_macro')['numero_de_operacoes'].pct_change(12) * 100

    df_selic = consultar_sgs(4189, 'selic_media')
    df_juros = consultar_sgs(20725, 'juros_giro')
    df_macro = pd.merge(df_selic, df_juros, on='ano_mes', how='inner').sort_values('ano_mes').reset_index(drop=True)
    df_macro['spread_bancario'] = df_macro['juros_giro'] - df_macro['selic_media']
    df_macro['selic_lag3'] = df_macro['selic_media'].shift(3)
    df_macro['selic_lag6'] = df_macro['selic_media'].shift(6)
    df_macro['juros_giro_lag3'] = df_macro['juros_giro'].shift(3)
    df_macro['juros_giro_lag6'] = df_macro['juros_giro'].shift(6)

    df_final = pd.merge(df_painel, df_macro, on='ano_mes', how='inner').dropna().reset_index(drop=True)

    # 4. Calibração de Markov e Partição OOT
    print('\n[4/5] Otimizando limiares endógenos via Persistência de Markov (P11 >= 0.70)...')
    corte_temporal = pd.to_datetime('2022-01-01')
    df_train_raw = df_final[df_final['data'] < corte_temporal].copy()
    df_test_raw = df_final[df_final['data'] >= corte_temporal].copy()

    limiares_markov = {}
    for setor in sorted(df_train_raw['setor_macro'].unique()):
        sub_tr = df_train_raw[df_train_raw['setor_macro'] == setor]
        tau_opt, p00, p11 = otimizar_limiar_markov(sub_tr, min_p11=0.70)
        limiares_markov[setor] = tau_opt
        print(f'  -> {setor:<20} | Limiar: {tau_opt:.2f}% | P00: {p00:.1%} | P11: {p11:.1%}')

    df_train_raw['alvo_inadimplencia_alta'] = (
        df_train_raw['taxa_inadimplencia_pct'] >= df_train_raw['setor_macro'].map(limiares_markov)
    ).astype(int)
    df_test_raw['alvo_inadimplencia_alta'] = (
        df_test_raw['taxa_inadimplencia_pct'] >= df_test_raw['setor_macro'].map(limiares_markov)
    ).astype(int)

    prevalencia_oot = df_test_raw['alvo_inadimplencia_alta'].mean()
    n_positivos = df_test_raw['alvo_inadimplencia_alta'].sum()
    print(f'Prevalência Real no Teste OOT (2022–2026): {prevalencia_oot:.2%} ({n_positivos} de {len(df_test_raw)} meses)')

    df_train = pd.get_dummies(df_train_raw, columns=['setor_macro'], drop_first=False, dtype=int)
    df_test = pd.get_dummies(df_test_raw, columns=['setor_macro'], drop_first=False, dtype=int)

    features = [
        'selic_media', 'selic_lag3', 'selic_lag6', 'juros_giro', 'juros_giro_lag3', 'juros_giro_lag6',
        'spread_bancario', 'ticket_medio_mil', 'crescimento_carteira_interanual', 'crescimento_operacoes_interanual'
    ] + [c for c in df_train.columns if c.startswith('setor_macro_')]

    X_train, y_train = df_train[features], df_train['alvo_inadimplencia_alta']
    X_test, y_test = df_test[features], df_test['alvo_inadimplencia_alta']

    # 5. Modelagem e SHAP
    print('\n[5/5] Treinando modelos supervisionados e computando Tree SHAP...')
    _, df_metricas, gbm_model = treinar_e_avaliar_modelos(X_train, y_train, X_test, y_test, prevalencia_oot)
    calcular_e_plotar_shap(gbm_model, X_test)

    print('\n' + '=' * 80)
    print('RESULTADOS OFICIAIS FORA DA AMOSTRA (OUT-OF-TIME 2022–2026):')
    print('=' * 80)
    print(df_metricas.to_string(index=False))
    print('\n[SUCESSO] Pipeline executado e artefatos exportados com êxito!')


if __name__ == '__main__':
    main()